# GRU Model Univariate


In this section we implement the GRU Model (Gated Recurrent Unit) using the **TimeSeriesDataset** approach with one-hot encoding.

The GRU (Gated Recurrent Unit) Forecaster is a gated recurrent neural network designed for time series forecasting. It uses one-hot encoding to identify individual series (1502 unique series), processing one series at a time. The model balances the complexity of LSTMs with the simplicity of vanilla RNNs, using two gates (reset and update) to control information flow.

With this appraoch each training sample represents a single series with its one-hot encoded identifier, allowing the model to learn series-specific patterns with fewer parameters than LSTM while still capturing long-term dependencies.

**Layer Breakdown**
- GRU Layers: 2 stacked GRU layers with gating mechanisms
- Hidden Size: 128 units per layer (default)
- Dropout: Applied between GRU layers (if >1 layer) and before final output
- Output Layer: Single fully connected layer producing 1-step forecast

**Advantages**

- Efficient Memory: Fewer parameters than LSTM (~25% reduction) - no separate cell state
- Better than RNN: Handles long-term dependencies better than vanilla RNN through gating
- Faster Training: Simpler gating mechanism (2 gates vs LSTM's 3) leads to faster computation
- Less Overfitting: Fewer parameters reduce overfitting risk on smaller datasets

**Limitations**
- Slightly Less Powerful: May underperform LSTM on very complex patterns requiring detailed memory control
- Still Sequential: Cannot be fully parallelized like CNNs or Transformers
- Gradient Issues: While better than vanilla RNN, can still face gradient problems on very long sequences (>100 steps)

**When to Use GRU**
- Dataset size is limited (prevents LSTM overfitting)
- Sequences are moderately long (10-50 timesteps)
- Pattern complexity is moderate (not requiring LSTM's fine-grained memory control)

## Model

In [ ]:
import torch
import torch.nn as nn

class GRUForecaster(nn.Module):
    """
    GRU model for MULTIVARIATE time series forecasting.
    Architecture: GRU -> Dropout -> GRU -> Dropout -> Fully Connected
    Takes multiple input features at each timestep.
    
    GRU is similar to LSTM but with fewer parameters (no cell state).
    Generally faster than LSTM while maintaining good performance.
    Uses reset and update gates instead of LSTM's input/forget/output gates.
    """
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        """
        Args:
            input_size: Number of input features (Value + year + month + one-hot)
            hidden_size: GRU hidden dimension
            num_layers: Number of GRU layers
            dropout: Dropout rate
        """
        super(GRUForecaster, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.input_size = input_size
        
        # GRU layers
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Dropout layer
        self.dropout = nn.Dropout(dropout)
        
        # Fully connected output layer
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        
        # GRU forward pass
        # gru_out: (batch_size, seq_length, hidden_size)
        # h_n: (num_layers, batch_size, hidden_size)
        gru_out, h_n = self.gru(x)
        
        # Take the output from the last time step
        last_output = gru_out[:, -1, :]  # Shape: (batch_size, hidden_size)
        
        # Apply dropout
        out = self.dropout(last_output)
        
        # Fully connected layer
        out = self.fc(out)  # Shape: (batch_size, 1)
        
        return out


## Model Results without Exogenous Features

In this section, the GRU model is evaluated based on temporal features (value, year, and month) and one-hot encoded series identifiers, without the incorporation of external economic indicators. This baseline approach enables assessment of how well series-specific patterns are captured using only historical information and temporal context. A 3-fold time series cross-validation strategy is employed to ensure robust performance evaluation and prevent data leakage.

### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Dropout | Hidden Size | Learning Rate | Num Layers | Duration |
|------:|----------------:|----------:|--------:|-----------:|--------------:|----------:|---------:|
| 0 | 0.34144 | 32 | 0.49866 | 256 | 0.00028 | 2 | 36.93s |
| 1 | 0.35373 | 128 | 0.24828 | 32 | 0.00613 | 2 | 31.72s |
| 2 | 0.34402 | 128 | 0.44691 | 32 | 0.00289 | 1 | 11.73s |


###Best Hyperparameters

Validation Loss: 0.34144

Parameters:
 - learning_rate: 0.00028
 - batch_size: 32
 - num_layers: 2
 - hidden_size: 256
 - dropout: 0.49866

#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/univariate/gru/fold1/fold_results.png)

#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/univariate/gru/fold2/fold_results.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

![Fold 3 Results](./img/univariate/gru/fold3/fold_results.png)

### Fold Results

| Fold | MSE | RMSE | MAE | R² | SMAPE |
|------|----------|----------|----------|------|-------|
| Fold 1 | 87873.62 | 296.43 | 123.32 | 0.8425 | 75.95% |
| Fold 2 | 98669.52 | 314.12 | 119.67 | 0.8032 | 73.65% |
| Fold 3 | 60667.18 | 246.31 | 96.98 | 0.8845 | 63.45% |
| **Average** | **82403.44 ± 19267.84** | **285.62 ± 34.40** | **113.33 ± 13.23** | **0.8434 ± 0.0386** | **71.35% ± 6.38%** |


### Average SMAPE Distribution Across Folds

| SMAPE Range | Percentage of Series | Number of Series (avg) |
|-------------|---------------------|------------------------|
| <10% | 5.9% ± 3.3% | 79 |
| 10-20% | 13.5% ± 2.0% | 180 |
| 20-30% | 14.2% ± 1.0% | 189 |
| 30-40% | 10.4% ± 0.3% | 139 |
| >40% | 56.0% ± 6.4% | 747 |


**Comparison with Baseline:**

The GRU univariate model with one-hot encoding achieves an average SMAPE of **71.35% ± 6.38%**, which is **0.91 percentage points lower** than the baseline 3-month rolling average (72.26% ± 7.06%). This indicates that the GRU model outperforms the baseline, demonstrating the effectiveness of gating mechanisms in capturing temporal dependencies for time series forecasting.

## Model Results with Exogenous Features


In this section, the GRU model is evaluated based on temporal features (value, year, and month), one-hot encoded series identifiers, and external economic indicators. This enhanced approach enables assessment of how exogenous features improve forecasting accuracy by incorporating domain-specific context beyond historical patterns. A 3-fold time series cross-validation strategy is employed to ensure robust performance evaluation and prevent data leakage.

### Optuna Hyperparameter Search Results

| Trial | Validation Loss | Batch Size | Dropout | Hidden Size | Learning Rate | Num Layers | Duration |
|------:|----------------:|----------:|--------:|-----------:|--------------:|----------:|---------:|
| 0 | 0.30380 | 64 | 0.14373 | 64 | 0.00528 | 1 | 43.56s |
| 1 | 0.33084 | 64 | 0.32340 | 64 | 0.00125 | 1 | 48.67s |
| 2 | 0.33907 | 64 | 0.20604 | 64 | 0.00026 | 3 | 37.17s |


###Best Hyperparameters

Validation Loss: 0.30380

Parameters:
  - learning_rate: 0.00528
  - batch_size: 64
  - num_layers: 1
  - hidden_size: 64
  - dropout: 0.14373

#### Analysis Fold 1 - Test period: 2024-10-01 until 2024-12-31

![Fold 1 Results](./img/univariate/gru_exo/fold1/fold_results.png)

#### Analysis Fold 2 - Test period: 2025-01-01 until 2025-03-31

![Fold 2 Results](./img/univariate/gru_exo/fold2/fold_results.png)

#### Analysis Fold 3 - Test period: 2025-07-01 until 2025-09-30

![Fold 3 Results](./img/univariate/gru_exo/fold3/fold_results.png)

### Fold Results

| Fold | MSE | RMSE | MAE | R² | SMAPE |
|------|----------|----------|----------|------|-------|
| Fold 1 | 77370.29 | 278.16 | 121.70 | 0.8613 | 68.44% |
| Fold 2 | 90845.45 | 301.41 | 126.08 | 0.8188 | 77.43% |
| Fold 3 | 63616.36 | 252.22 | 102.56 | 0.8789 | 65.92% |
| **Average** | **77610.71 ± 13794.79** | **277.26 ± 25.10** | **116.81 ± 12.40** | **0.8530 ± 0.0316** | **70.60% ± 5.99%** |

### Average SMAPE Distribution Across Folds

| SMAPE Range | Percentage of Series | Number of Series (avg) |
|-------------|---------------------|------------------------|
| <10% | 6.7% ± 4.7% | 90 |
| 10-20% | 10.6% ± 1.5% | 142 |
| 20-30% | 13.5% ± 0.4% | 181 |
| 30-40% | 10.7% ± 2.0% | 144 |
| >40% | 58.4% ± 7.6% | 778 |

**Comparison with Baseline and Univariate Model:**

The GRU model incorporating exogenous features achieves an average SMAPE of 70.60% ± 5.99%, whereas the univariate GRU model without exogenous inputs attains a lower SMAPE of 71.35% ± 6.38%. This indicates that the univariate model slightly underperforms the exogenous variant by 0.75 percentage points.

Both GRU models demonstrate a slightly superior performance compared to the baseline 3-month rolling average (72.26% ± 7.06%). The exogenous GRU model outperforms the baseline by 1.66 percentage points, while the univariate model outperforms it by 0.91 percentage points. The slightly stronger accuracy of the exogenous model suggests that incorporating external economic indicators provides additional predictive value for this dataset. 